# Somo la 05 - RAG ya Wakala


## Setup

Daftari hili linaonyesha muundo wa Agentic RAG (Retrieval-Augmented Generation) kwa kutumia Microsoft Agent Framework.

**Mahitaji ya awali:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — kituo chako cha huduma ya Azure AI Search
- `AZURE_SEARCH_API_KEY` — ufunguo wako wa API wa Azure AI Search
- Utekelezaji wa Azure OpenAI umewekwa kupitia vigezo vya mazingira
- Azure CLI imethibitishwa (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG ni Nini?

RAG ya kawaida hufuata mchakato uliowekwa: chukua nyaraka, kisha tengeneza jibu. **Agentic RAG** inaenda zaidi kwa kumpa wakala uhuru wa kuamua **lini** na **vipi** kupata taarifa.

Kwa Agentic RAG, wakala anaweza:
- **Kuamua** kama kunahitaji kupata taarifa kabla ya kujibu swali
- **Kuchagua** chanzo cha data au chombo cha kuuliza
- **Kutathmini** matokeo yaliyopatikana na kufanya upokeaji wa taarifa zaidi kama jaribio la kwanza halitoshi
- **Kuchanganya** taarifa kutoka hatua nyingi za upokeaji kuwa jibu linaloeleweka vizuri

Hii inafanya wakala kuwa rahisi kubadilika na sahihi zaidi ikilinganishwa na mchakato wa kawaida wa kupata-kisha-kutengeneza.


## Kuunda Zana ya Utafutaji

Katika Agentic RAG, vyanzo vya data vya nje vimefungwa kama **zana** ambazo wakala anaweza kuita anapohitaji. Hii inamwezesha wakala kutendea upokeaji kama tendo lingine analoweza kuchukua, badala ya hatua ya lazima.

Hapa chini tunafafanua hifadhi ya maarifa ya usafiri na kuionesha kama zana ambayo wakala anaweza kuitisha kutafuta habari za maeneo ya kwenda.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Kujenga Wakala wa RAG

Sasa tunaunda wakala ambaye ameamriwa **daima apate taarifa kabla ya kujibu**. Wakala hutumia chombo cha `search_travel_knowledge` kudumisha majibu yake kwa msingi wa hifadhidata ya maarifa badala ya kutegemea data yake ya mafunzo.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Urejeshaji wa Msururu — Mfano wa Mtayarishaji-Mhakiki

Faida kuu ya Agentic RAG ni **urejeshaji wa msururu**. wakala anaweza kufanya mizunguko kadhaa ya utafutaji ili kuthibitisha, kuboresha, au kuongeza kwenye matokeo yake ya awali — sawa na mtiririko wa kazi wa "mtayarishaji-mhakiki":

1. **Hatua ya Mtayarishaji**: Wakala anarudisha taarifa za awali na kuandika jibu.
2. **Hatua ya Mhakiki**: Wakala hufanya urejeshaji zaidi kuthibitisha maelezo au kujaza mapengo.

Hapo chini, wakala anaulizwa swali linalohitaji kulinganisha maeneo mbalimbali, linalomsukuma kufanya utafutaji mara kadhaa.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Muhtasari

Katika somo hili umejifunza jinsi ya kujenga mfumo wa **Agentic RAG** ukitumia Microsoft Agent Framework:

- **Agentic RAG** huwafanya maajenti kuamua wenyewe wakati wa kupata taarifa, kufanya upatikanaji kuwa wa mabadiliko badala ya wa kudumu.
- **Zana kama vyanzo vya data**: Maktaba za maarifa za nje (kama Azure AI Search) zimefungwa kama zana ambazo maajenti wanaweza kuitumia.
- **Upatikanaji wa mara kwa mara**: Mtindo wa mtengenezaji-mhakiki huwafanya maajenti kufanya mizunguko mingi ya upatikanaji — kutafuta, kuthibitisha, na kuboresha — kabla ya kutoa jibu la mwisho.

Katika uzalishaji, ungebadilisha `TRAVEL_KNOWLEDGE_BASE` ya kumbukumbu ndani na kiashiria halisi cha Azure AI Search kushughulikia upatikanaji mkubwa wa hati za usafiri.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kiarife**:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kwa usahihi, tafadhali fahamu kuwa tafsiri za otomatiki zinaweza kuwa na makosa au upotoshaji. Nyaraka ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya mtaalamu wa binadamu inapendekezwa. Hatubebwi na dhima kwa kutoelewana au tafsiri zisizo sahihi zinazotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
